# PyTorch Lightning 完整学习笔记（下）

> 本笔记涵盖工程化工具、高级训练范式、分布式训练基础设施与源码级架构。  
> 掌握这部分，你将能从“会用 Lightning”跃升至“精通并自定义 Lightning”，应对大规模、复杂项目的挑战。

---

## 第六章：工程化工具

### 6.1 Logger 体系 —— 统一管理实验指标

Lightning 将日志系统分为三层：`self.log` → `LoggerConnector` → 具体 `Logger`。  
你只需在模型里调用 `self.log`，无需关心后端。

#### 6.1.1 `self.log` 详细用法

```python
def training_step(self, batch, batch_idx):
    loss = ...
    # ================= self.log 参数说明 =================
    self.log(
        "train_loss",      # 指标名称，可使用 "train/loss" 进行分组
        loss,              # 值：Tensor（标量）、float、int
        prog_bar=True,     # 是否显示在进度条上
        logger=True,       # 是否发送给 Logger（TensorBoard/WandB等）
        on_step=True,      # 是否在每一步都记录（训练时建议 True，观察细粒度）
        on_epoch=True,     # 是否在 epoch 结束时记录聚合值（验证时建议 True）
        sync_dist=True,    # 多卡训练时，是否跨卡同步（验证/测试时必须 True）
        reduce_fx="mean"   # epoch 聚合函数，默认 torch.mean，也可用 sum/max 等
    )
    return loss
```

**最佳实践**：  
- 训练指标：`on_step=True, on_epoch=True`，同时保留步骤和 epoch 曲线。  
- 验证/测试指标：`on_step=False, on_epoch=True, sync_dist=True`，确保多卡下指标正确。  
- 命名使用 `/` 分隔，如 `"val/acc"`, `"train/loss"`，TensorBoard 会自动分组。

#### 6.1.2 使用内置 Logger

```python
from lightning.pytorch.loggers import TensorBoardLogger, WandbLogger, CSVLogger

# TensorBoard
tb_logger = TensorBoardLogger(
    save_dir="logs",       # 根目录
    name="my_experiment"   # 实验名，子目录
)
# 启动：tensorboard --logdir logs

# Weights & Biases（云端实验追踪）
wandb_logger = WandbLogger(
    project="mnist_classification",  # 项目名
    name="run_lr_1e-3_h256",        # 本次运行名
    log_model="all"                  # 上传模型 checkpoint
)

# CSV（离线分析）
csv_logger = CSVLogger(save_dir="logs", name="csv_run")

# 可同时使用多个 Logger
trainer = pl.Trainer(logger=[tb_logger, wandb_logger])
```

#### 6.1.3 `log_dict` 批量记录

```python
# 代替多次 self.log
metrics = {
    "train/loss": loss,
    "train/acc": acc,
    "lr": self.optimizers().param_groups[0]["lr"]
}
self.log_dict(metrics, prog_bar=True)
# 参数与 self.log 相同，可全局设置 sync_dist, on_step 等
```

---

### 6.2 TorchMetrics —— 分布式安全的指标计算

手动计算指标在多卡下易出错，TorchMetrics 自动管理**状态累积、跨卡同步、重置**。

#### 6.2.1 基本用法

```python
from torchmetrics import Accuracy

class LitModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        # 训练和验证指标必须分开，避免状态污染
        self.train_acc = Accuracy(task="multiclass", num_classes=10)
        self.val_acc = Accuracy(task="multiclass", num_classes=10)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        # 调用 metric 对象，内部自动 update 状态
        acc = self.train_acc(logits, y)
        self.log("train_acc", acc, on_step=True, on_epoch=True)
        return F.cross_entropy(logits, y)

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        acc = self.val_acc(logits, y)
        self.log("val_acc", acc, on_epoch=True, sync_dist=True)
```

#### 6.2.2 常用指标一览

```python
from torchmetrics import (
    Accuracy, Precision, Recall, F1Score,
    AUROC, ConfusionMatrix,
    MeanSquaredError, MeanAbsoluteError,
)
# 分类
acc = Accuracy(task="binary", num_classes=1)  # 二分类用 binary
f1 = F1Score(task="multiclass", num_classes=10)
auc = AUROC(task="multiclass", num_classes=10)

# 回归
mse = MeanSquaredError()
mae = MeanAbsoluteError()
```

#### 6.2.3 使用 MetricCollection 管理多个指标

```python
from torchmetrics import MetricCollection

metrics = MetricCollection({
    "acc": Accuracy(task="multiclass", num_classes=10),
    "f1": F1Score(task="multiclass", num_classes=10)
})

def validation_step(self, batch, batch_idx):
    x, y = batch
    logits = self(x)
    results = metrics(logits, y)  # 返回字典 {"acc": ..., "f1": ...}
    self.log_dict(results, on_epoch=True, sync_dist=True)
```

---

### 6.3 Checkpoint 机制详解

Lightning 的 checkpoint 保存完整训练状态：模型权重、优化器、调度器、epoch、超参数等。

#### 6.3.1 自动保存最佳模型

```python
from lightning.pytorch.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",       # 监控的指标名
    mode="min",               # "min" 或 "max"
    save_top_k=3,             # 保留最好的 k 个模型
    save_last=True,           # 同时保存最后一个 epoch 模型（断点续训用）
    dirpath="./checkpoints",
    filename="best-{epoch:02d}-{val_loss:.2f}"
)
trainer = pl.Trainer(callbacks=[checkpoint_callback])
```

#### 6.3.2 断点续训

```python
# 从上次中断的 checkpoint 继续训练
trainer.fit(model, datamodule=dm, ckpt_path="last.ckpt")
```

#### 6.3.3 加载模型用于推理

```python
# 直接加载完整模型（需要原类定义）
model = LitModel.load_from_checkpoint("best.ckpt")
# 此时 self.hparams 已自动恢复
model.eval()
pred = model(some_input)
```

#### 6.3.4 自定义保存内容

```python
def on_save_checkpoint(self, checkpoint):
    # 将 tokenizer 等非模型状态存入 checkpoint
    checkpoint["my_tokenizer"] = self.tokenizer

def on_load_checkpoint(self, checkpoint):
    self.tokenizer = checkpoint["my_tokenizer"]
```

---

### 6.4 Prediction / Inference 体系

Lightning 提供统一的分布式预测接口。

#### 6.4.1 `predict_step` 与 `trainer.predict`

```python
def predict_step(self, batch, batch_idx, dataloader_idx=0):
    x, _ = batch  # 忽略标签
    logits = self(x)
    return torch.argmax(logits, dim=1)  # 返回类别索引

# 触发预测
predictions = trainer.predict(model, datamodule=dm)
# predictions 是列表，每个元素为一个 batch 的预测结果（按顺序拼接）
```

#### 6.4.2 多预测集

```python
# DataModule 中
def predict_dataloader(self):
    return [loader_A, loader_B]

# predict_step 中根据 dataloader_idx 区分
def predict_step(self, batch, batch_idx, dataloader_idx=0):
    if dataloader_idx == 0:
        # 处理数据集 A
        ...
    else:
        ...
```

---

### 6.5 模型导出与部署

Lightning 本身是训练框架，部署需转换格式。

#### 6.5.1 导出原生 PyTorch

```python
torch.save(model.state_dict(), "model.pt")
# 加载
model.load_state_dict(torch.load("model.pt"))
```

#### 6.5.2 TorchScript

```python
scripted = model.to_torchscript()
torch.jit.save(scripted, "model.pt")
# 可脱离 Python 环境运行
```

#### 6.5.3 ONNX

```python
input_sample = torch.randn(1, 3, 224, 224)
model.to_onnx("model.onnx", input_sample, export_params=True)
# 后续可用 ONNX Runtime / TensorRT 加速
```

#### 6.5.4 实际部署路线

**Web API 服务**：`Lightning Checkpoint → PyTorch → FastAPI → Docker → Kubernetes`  
**大模型推理**：`Lightning Checkpoint → 转为 Hugging Face 格式 → vLLM / TGI`  
**极致性能**：`PyTorch/ONNX → TensorRT → Triton Inference Server`

---

## 第七章：高级训练范式

### 7.1 Manual Optimization（手动优化）

关闭 Trainer 的自动反向/更新，完全掌控训练循环。

```python
class ManualModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.automatic_optimization = False  # 关键开关

    def training_step(self, batch, batch_idx):
        opt = self.optimizers()  # 获取优化器（单或多）
        opt.zero_grad()

        loss = ...  # 前向与损失
        self.manual_backward(loss)  # 必须使用 manual_backward
        opt.step()

        # 手动梯度累积示例
        # if (batch_idx + 1) % 4 == 0:
        #     opt.step()
        #     opt.zero_grad()

        self.log("loss", loss)
        return loss
```

**`manual_backward` 的重要性**：它内部会经过 Strategy 和 Precision 插件，正确处理混合精度和分布式同步。永远不要直接调用 `loss.backward()`。

---

### 7.2 多优化器与差异化学习率

```python
def configure_optimizers(self):
    # 分别优化不同模块
    opt_encoder = torch.optim.AdamW(self.encoder.parameters(), lr=1e-5)
    opt_head = torch.optim.AdamW(self.head.parameters(), lr=1e-3)
    # 也可加上 scheduler
    scheduler_enc = torch.optim.lr_scheduler.StepLR(opt_encoder, step_size=5)
    return [opt_encoder, opt_head], [scheduler_enc, ...]

def training_step(self, batch, batch_idx):
    opt_enc, opt_head = self.optimizers()
    # 手动模式下各自 zero_grad / backward / step
```

---

### 7.3 GAN 训练完整示例

```python
class GANLightning(pl.LightningModule):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.automatic_optimization = False
        self.generator = nn.Sequential(...)
        self.discriminator = nn.Sequential(...)
        self.latent_dim = latent_dim

    def configure_optimizers(self):
        g_opt = torch.optim.Adam(self.generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
        d_opt = torch.optim.Adam(self.discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
        return [g_opt, d_opt]

    def training_step(self, batch, batch_idx):
        g_opt, d_opt = self.optimizers()
        real = batch
        z = torch.randn(real.size(0), self.latent_dim, device=self.device)

        # ----- 训练判别器 D -----
        with torch.no_grad():
            fake = self.generator(z)
        real_pred = self.discriminator(real)
        fake_pred = self.discriminator(fake)
        d_loss = -torch.mean(real_pred) + torch.mean(fake_pred)  # Wasserstein loss 简化

        d_opt.zero_grad()
        self.manual_backward(d_loss)
        d_opt.step()

        # ----- 训练生成器 G -----
        z = torch.randn(real.size(0), self.latent_dim, device=self.device)
        fake = self.generator(z)
        fake_pred = self.discriminator(fake)
        g_loss = -torch.mean(fake_pred)

        g_opt.zero_grad()
        self.manual_backward(g_loss)
        g_opt.step()

        self.log_dict({"d_loss": d_loss, "g_loss": g_loss}, prog_bar=True)
```

---

### 7.4 RL 训练（PPO / Actor-Critic 概念）

```python
class PPOModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.automatic_optimization = False
        self.actor = ...   # 策略网络
        self.critic = ...  # 价值网络

    def configure_optimizers(self):
        actor_opt = torch.optim.Adam(self.actor.parameters(), lr=3e-4)
        critic_opt = torch.optim.Adam(self.critic.parameters(), lr=1e-3)
        return [actor_opt, critic_opt]

    def training_step(self, batch, batch_idx):
        # 假设 batch 已包含采样的轨迹（states, actions, rewards, old_log_probs）
        actor_opt, critic_opt = self.optimizers()
        states, actions, rewards, old_log_probs = batch

        # 计算当前策略的 log_probs 和 value
        log_probs = self.actor.evaluate(states, actions)
        values = self.critic(states)
        advantages = rewards - values.detach()

        # Actor loss (PPO clipped)
        ratio = (log_probs - old_log_probs).exp()
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 0.8, 1.2) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()

        # Critic loss
        critic_loss = F.mse_loss(values, rewards)

        # 更新 Actor
        actor_opt.zero_grad()
        self.manual_backward(actor_loss)
        actor_opt.step()

        # 更新 Critic
        critic_opt.zero_grad()
        self.manual_backward(critic_loss)
        critic_opt.step()

        self.log_dict({"actor_loss": actor_loss, "critic_loss": critic_loss})
```

**RLHF 本质**：Actor = LLM，Critic = Reward Model，训练结构依然是多优化器手动控制。

---

## 第八章：分布式与大模型训练基础设施

### 8.1 DDP（Distributed Data Parallel）

每张卡持有完整模型副本，数据切片，反向传播后 AllReduce 平均梯度。  
**解决算力，不减少单卡显存**。

```python
trainer = pl.Trainer(
    accelerator="gpu",
    devices=4,
    strategy="ddp",        # 自动启动多进程，处理 DistributedSampler
    precision="bf16-mixed"
)
```

**注意事项**：验证指标记录必须添加 `sync_dist=True`，否则只记录当前卡的结果。

---

### 8.2 FSDP（Fully Sharded Data Parallel）

将参数、梯度、优化器状态切分到多卡，显著降低单卡显存。适用于 7B~70B+ 模型。

```python
trainer = pl.Trainer(
    accelerator="gpu",
    devices=8,
    strategy="fsdp",
    precision="bf16-mixed",
    gradient_clip_val=1.0
)
```

**原理**：前向时通过 AllGather 拼出当前层参数，计算后立即释放，显存中仅保留当前计算所需切片。

---

### 8.3 DeepSpeed

微软开发的极致显存优化库，提供 ZeRO 阶段 1/2/3，支持 CPU/NVMe Offload。

```python
from lightning.pytorch.strategies import DeepSpeedStrategy

strategy = DeepSpeedStrategy(
    stage=2,                # ZeRO-2：切分优化器状态 + 梯度
    offload_optimizer=True, # 优化器状态卸载到 CPU
)

trainer = pl.Trainer(
    accelerator="gpu",
    devices=8,
    strategy=strategy,
    precision="bf16-mixed"
)
```

**选择**：PyTorch 官方逐步推荐 FSDP，但 DeepSpeed 在百亿参数以上仍有生态优势。

---

### 8.4 混合精度训练（AMP/BF16）

- `precision="16-mixed"`：自动混合精度（FP16），需 GradScaler 防止梯度下溢，Lightning 自动处理。  
- `precision="bf16-mixed"`：更稳定，无需 scaler，推荐 A100/H100/4090 使用。

```python
trainer = pl.Trainer(
    precision="bf16-mixed",   # 省显存 40%~50%，速度更快
)
```

内部机制：
1. 前向：`autocast` 自动选择合适的精度。
2. 反向：FP16 时 `scaler.scale(loss).backward()`；BF16 时直接 `loss.backward()`。
3. 优化器：FP16 时 `scaler.step(optimizer)` 并在更新前 `unscale_`。

---

### 8.5 Fabric —— 保留 PyTorch 自由度，获取分布式能力

`Trainer` 是全自动框架，`Fabric` 则是半自动工具。适用于需要手写训练循环但想利用分布式/精度的场景（RL、GAN、元学习）。

```python
from lightning import Fabric

fabric = Fabric(accelerator="gpu", devices=4, precision="bf16-mixed")
model, optimizer = fabric.setup(model, optimizer)
train_loader = fabric.setup_dataloaders(train_loader)

model.train()
for epoch in range(max_epochs):
    for batch in train_loader:
        optimizer.zero_grad()
        loss = model(batch)
        fabric.backward(loss)   # 替代 loss.backward()
        optimizer.step()
```

`fabric.backward()` 内部会处理梯度缩放与分布式同步。  
**适用场景**：完全自定义的训练逻辑，如多步更新、采样与训练交替等。

---

## 第九章：源码级架构（Lightning 内核）

Lightning 运行时由四层抽象构成：**Loop、Strategy、Precision、Accelerator**。

### 9.1 Loop System —— 控制流引擎

层次化状态机：`FitLoop → TrainingEpochLoop → TrainingBatchLoop`。

- `FitLoop`：控制 epoch 循环、停止条件、EarlyStopping 检查、checkpoint 保存。
- `TrainingEpochLoop`：遍历 DataLoader，触发 epoch 级 Hook。
- `TrainingBatchLoop`：处理单个 batch，依次调用 Hook：`training_step` → `backward` → `optimizer_step` 等。

自定义 Loop 可替换默认训练循环，实现如 PPO 等非标准流程。

### 9.2 Strategy System —— 分布式抽象层

不同分布式后端的实现封装在 `Strategy` 中，统一接口：

- `strategy.backward(loss)`
- `strategy.optimizer_step(optimizer)`
- `strategy.setup(model)`
- `strategy.save_checkpoint()` / `load_checkpoint()`

继承结构：
```
Strategy
├── SingleDeviceStrategy
├── DDPStrategy
├── FSDPStrategy
└── DeepSpeedStrategy
```

**重要性**：`self.manual_backward(loss)` 最终调用的是 `Trainer.strategy.backward(loss)`，因此分布式/精度逻辑能透明执行。

### 9.3 Precision Plugin —— 精度控制

负责管理 `autocast` 上下文和 `GradScaler`。  
- `precision="16-mixed"`：前向使用 `autocast`，反向使用 `GradScaler`。  
- `precision="bf16-mixed"`：仅 `autocast`，无 scaler。

在 `Strategy.backward()` 内部调用 `Precision.backward()`，形成嵌套。

### 9.4 Accelerator —— 硬件抽象

处理具体硬件设备初始化与操作：
- `CUDAAccelerator`：`torch.cuda.set_device`
- `TPUAccelerator`：`xm.xla_device`
- `CPUAccelerator`、`MPSAccelerator`

Trainer 初始化时，`AcceleratorConnector` 根据参数选择并设置相应设备。

### 9.5 Callback Connector —— 回调调度器

负责 Callback 的注册、排序与生命周期调度。  
Trainer 在关键节点遍历所有已注册 Callback，并先于 `LightningModule` 的同名 Hook 执行。

### 9.6 完整调用链路（以单个训练 batch 为例）

```
TrainingBatchLoop.run(batch)
  → Callback.on_train_batch_start
  → Module.on_train_batch_start
  → Module.training_step                          (返回 loss)
  → Callback.on_before_backward(loss)
  → Module.on_before_backward(loss)
  → Strategy.backward(loss)                        (内部调用 Precision.backward)
  → Callback.on_after_backward
  → Module.on_after_backward
  → Callback.on_before_optimizer_step(opt)
  → Module.on_before_optimizer_step(opt)
  → Strategy.optimizer_step(opt)                   (内部调用 Precision.optimizer_step)
  → Module.on_before_zero_grad(opt)
  → opt.zero_grad()
  → Callback.on_train_batch_end(outputs)
  → Module.on_train_batch_end(outputs)
```

---

## 总结

你现在已经掌握了 PyTorch Lightning 的完整知识体系：

- **工程化工具**：Logger、TorchMetrics、Checkpoint、Prediction、部署。
- **高级训练**：手动优化、多优化器、GAN、RL/PPO、RLHF。
- **分布式基础设施**：DDP、FSDP、DeepSpeed、AMP/BF16、Fabric。
- **源码架构**：Loop、Strategy、Precision、Accelerator、Callback Connector 及全链路调用。

